# NB02 · ¿Con qué modelo? — modelos, prefijos, normalización, métrica

**D09 ya está decidida**: compiten `gemini-embedding-2`, `jinaai/jina-embeddings-v3` e `ibm-granite/granite-embedding-311m-multilingual-r2`, con la **plantilla congelada en A0** (la columna `text` tal cual).

El notebook empieza por el **paso 0**: medir cuánto texto ve realmente cada candidato. Es la evidencia que decide **D07** (¿hace falta chunking?) y la que confirma que la ventana de los tres cubre el catálogo.

⚠️ **Las longitudes de NB01 no sirven aquí.** Allí se midió `text` en **palabras** (p50 = 150); la ventana de un modelo se mide en **piezas de su vocabulario de subpalabras**, y un código como `160x200` se lleva varias. La única cuenta válida es la del tokenizador de cada modelo, y por eso se descarga aquí.

| Marca | Corpus | Fichero |
|---|---|---|
| 🔬 **MUESTRA** | 1.500 registros | `catalogo_muestra.csv` |
| 📚 **COMPLETO** | 15.000 registros | `catalogo_productos.csv` |

> 📄 **Cada celda de código declara sus datos en la primera línea, incluidas las que no leen ninguno.** El mismo modelo baja de nDCG al pasar de 1.500 candidatos a 15.000 sin que nada haya empeorado, así que saber qué corpus se está mirando cambia la lectura de cualquier métrica. Además de los dos catálogos aparecen `consultas_desarrollo.csv` (8 consultas con juicios), `relevancias_desarrollo.csv` (248 juicios de relevancia) y `consultas_evaluacion.csv` (12 ciegas, sin juicios).

**El cambio de corpus ocurre en la sección I**: todo lo anterior mide sobre la muestra.

In [ ]:
# 📄 DATOS · carga catalogo_muestra.csv (1.500) y catalogo_productos.csv (15.000)
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path("..") / "src"))

import pandas as pd
from dotenv import load_dotenv
from transformers import logging as hf_logging

from aurum.embeddings import load_hub_tokenizer, token_length_report, token_lengths

hf_logging.set_verbosity_error()
# Carga HF_TOKEN y GEMINI_API_KEY en el entorno del proceso. El fichero .env no se
# imprime ni se versiona: solo se leen los valores desde os.environ.
load_dotenv(Path("..") / ".env")

DATA = Path("..") / "data"
muestra = pd.read_csv(DATA / "catalogo_muestra.csv")
completo = pd.read_csv(DATA / "catalogo_productos.csv")

print(f"🔬 MUESTRA : {len(muestra):>6} registros")
print(f"📚 COMPLETO: {len(completo):>6} registros")
print(f"HF_TOKEN cargado: {bool(os.environ.get('HF_TOKEN'))}")

## A.1 · Los tres candidatos de D09

Datos de cada uno, contrastados contra su *model card* y su `config.json` (no contra la memoria de nadie). `gemini-embedding-2` es API, así que no tiene tokenizador descargable: su ventana se toma de la documentación oficial.

| Modelo | Dim nativa | Ventana | Contrato de entrada | Acceso |
|---|---:|---:|---|---|
| `gemini-embedding-2` | 3.072 (MRL 128–3.072) | 8.192 | instrucción **en el prompt** — no admite `task_type` | API de Google (`GEMINI_API_KEY`) |
| `jinaai/jina-embeddings-v3` | 1.024 (MRL 32–1.024) | 8.192 | **adaptadores LoRA por tarea**: `retrieval.query` · `retrieval.passage` · `text-matching` | HF abierto, licencia **`cc-by-nc-4.0`** ⚠️ |
| `ibm-granite/granite-embedding-311m-multilingual-r2` | 768 | 32.768 | por confirmar en su model card | HF abierto (Apache-2.0) |

In [ ]:
# 📄 DATOS · ninguno: solo descarga los tokenizadores del Hub
CANDIDATOS_HF = {  # espejo de config.yaml -> nb02_modelo.d09_modelos
    "jina-v3": ("jinaai/jina-embeddings-v3", 8192),
    "granite-311m-r2": ("ibm-granite/granite-embedding-311m-multilingual-r2", 32768),
}
VENTANA_GEMINI = 8192  # gemini-embedding-2, según la documentación de la API

# `load_hub_tokenizer` baja solo el tokenizer.json: jina-v3 lleva código propio en
# el repo y AutoTokenizer exigiría trust_remote_code, que importa torch. Para
# contar tokens basta el vocabulario.
tokenizadores = {}
for alias, (repo, ventana) in CANDIDATOS_HF.items():
    try:
        tokenizadores[alias] = (
            load_hub_tokenizer(repo, token=os.environ.get("HF_TOKEN")),
            ventana,
        )
        print(f"✅ {alias}: tokenizador descargado")
    except Exception as error:  # sin red, repo gated o token sin permiso
        print(f"⛔ {alias}: {type(error).__name__} — {str(error).splitlines()[0]}")

## A.2 · 🔬 Longitud en tokens sobre la muestra

`pct_supera_ventana` es el número que decide **D07**. `chars_por_token` mide cuánto se aleja la cuenta real de una estimación en caracteres.

In [ ]:
# 📄 DATOS · 🔬 catalogo_muestra.csv (1.500 fichas)
token_length_report(muestra["text"], tokenizadores)

## A.3 · 📚 La misma medición sobre el catálogo completo

⏱️ Tarda ~30 s: tokeniza 15.000 registros.

In [ ]:
# 📄 DATOS · 📚 catalogo_productos.csv (15.000 fichas)
informe_completo = token_length_report(completo["text"], tokenizadores)
informe_completo

## A.4 · 📚 Cuántos registros del catálogo se truncarían con cada tamaño de ventana

La pregunta de fondo de D07 no es *"¿se trunca?"* sino *"¿a partir de qué ventana deja de truncarse?"*. Esta tabla la responde de una vez para cualquier modelo, presente o futuro.

In [ ]:
# 📄 DATOS · 📚 catalogo_productos.csv (15.000 fichas)
alias_referencia = next(iter(tokenizadores))
longitudes = token_lengths(completo["text"], tokenizadores[alias_referencia][0])

pd.DataFrame([
    {
        "ventana": ventana,
        "registros_que_la_superan": int((longitudes > ventana).sum()),
        "pct": round(100 * float((longitudes > ventana).mean()), 2),
    }
    for ventana in (128, 512, 1024, 2048, 8192)
]).assign(tokenizador=alias_referencia)

## A.5 · 📚 `gemini-embedding-2`: medición contra la API

Los dos modelos locales se miden con su tokenizador descargado. Gemini no publica el suyo, pero la API expone `count_tokens` **para el propio modelo de embeddings** — así que no hay que estimar nada ni usar un modelo generativo como sustituto.

⚠️ **Es una petición de red por registro**, así que se mide un subconjunto: los **50 más largos** en caracteres —los únicos que podrían acercarse a la ventana— más **100 al azar** para el ratio `chars_por_token`. Con un máximo local de 1.972 tokens contra una ventana de 8.192, el margen es de 4×: no hace falta más precisión para responder a D07.

Sin `GEMINI_API_KEY` la celda se salta sin romper el notebook: el corrector puede ejecutar el resto sin clave.

In [ ]:
# 📄 DATOS · 📚 catalogo_productos.csv → subconjunto de 150 fichas (una llamada por ficha)
from aurum.embeddings import CountingTokenizer, gemini_token_counter

MODELO_GEMINI = "gemini-embedding-2"
N_MAS_LARGAS, N_AZAR = 50, 100

longitud_chars = completo["text"].fillna("").str.len()
mas_largas = completo.loc[longitud_chars.nlargest(N_MAS_LARGAS).index, "text"]
al_azar = completo["text"].sample(N_AZAR, random_state=42)
subconjunto = pd.concat([mas_largas, al_azar]).drop_duplicates()

clave = os.environ.get("GEMINI_API_KEY")
if not clave:
    print("⏭️  Sin GEMINI_API_KEY: se omite la medición de gemini-embedding-2")
else:
    gemini = CountingTokenizer(gemini_token_counter(MODELO_GEMINI, api_key=clave))
    informe_gemini = token_length_report(subconjunto, {MODELO_GEMINI: (gemini, VENTANA_GEMINI)})
    display(informe_gemini)

### A.5b · 📚 El mismo subconjunto con los tres tokenizadores

Comparar los tres sobre **los mismos registros** es lo que exige la Regla 2: si cada modelo se midiera sobre un corpus distinto, las columnas no serían comparables entre sí.

In [ ]:
# 📄 DATOS · 📚 el mismo subconjunto de 150 fichas de catalogo_productos.csv
if clave:
    todos = {MODELO_GEMINI: (gemini, VENTANA_GEMINI), **tokenizadores}
    display(token_length_report(subconjunto, todos))

## A.6 · Cómo se lee esto para D07

El chunking solo tiene sentido si el modelo **no puede leer el registro entero**. Con `pct_supera_ventana = 0` en los tres candidatos, no hay información que se pierda por truncado: queda descartado **por medición**, no por falta de tiempo.

Consecuencia en la base vectorial: el punto sigue siendo `record_id` (1:1 producto↔vector), el esquema de NB04 se mantiene simple y la idempotencia no necesita borrar chunks huérfanos.

---

# B · Codificación de los tres candidatos

Hasta aquí se ha medido **cuánto texto ve** cada modelo. Ahora se codifica de verdad, sobre `catalogo_muestra.csv`: la muestra existe justo para esto y el catálogo completo solo se ingiere en la ejecución final.

### Qué se codifica y qué no

| Eje de D10 | ¿Obliga a recodificar? | Cómo se barre |
|---|---|---|
| **Modelo** | Sí | 3 codificaciones |
| **Contrato de entrada** (con/sin prefijos) | Sí | ×2 **solo** en los modelos que tienen contrato |
| **Dimensión (MRL)** | No | Truncar + renormalizar los mismos vectores |
| **Normalización L2** | No | Post-proceso |
| **Métrica** (`cosine`·`dot`·`l2`) | No | Cambia el buscador, no los vectores |

Los tres últimos ejes son **gratis**, y por eso el barrido de la sección C cubre 15 configuraciones sin pagar 15 codificaciones.

### ⏱️ Lo que cuesta en esta máquina

4 núcleos, sin GPU. `jina-embeddings-v3` son 572M de parámetros: en `float32` ocupa ~2,3 GB de los 7,9 disponibles, así que **los modelos se cargan y se liberan de uno en uno**, y la primera pasada son decenas de minutos.

La segunda es instantánea: `encode_corpus` cachea en `artifacts/embeddings/` con un `.json` que lleva `model_id`, dimensión, dtype y **SHA-256 del corpus**. Si el texto cambia (al pasar de A0 a otra plantilla en NB03), la huella cambia y la caché se invalida sola — lo que impide comparar en silencio vectores de dos textos distintos.

In [ ]:
# 📄 DATOS · 🔬 catalogo_muestra.csv (1.500) es el corpus de aquí en adelante
#            + consultas_desarrollo.csv (8) · relevancias_desarrollo.csv (248 juicios)
#            + consultas_evaluacion.csv (12 ciegas, sin juicios)
import gc
import time

import numpy as np
import torch

from aurum.busqueda import DenseRetriever, rank_queries_dense
from aurum.embeddings import (
    GeminiEncoder,
    SentenceTransformerEncoder,
    api_cost_report,
    drift_check,
    encode_corpus,
    measure_encode_latency,
    safe_l2_normalize,
    truncate_dim,
    vector_health,
)
from aurum.evaluacion import (
    apply_tolerance_rule,
    evaluate_rankings,
    formulation_consistency,
    qrels_from_judgements,
)
from aurum.graficas import (
    plot_contract_delta,
    plot_dimension_curve,
    plot_metric_comparison,
)

torch.set_num_threads(4)  # los 4 núcleos físicos de la máquina

CORPUS_ID = "catalogo_muestra"   # condición 3 del plan
PLANTILLA = "A0"                 # congelada: la columna `text` tal cual
CAMPO = "text"
TOP_K = 10
BATCH_LOCAL = 8                  # 8 GB de RAM con un modelo de 572M cargado
CACHE = Path("..") / "artifacts" / "embeddings"
TOLERANCIA_D09B = 0.02           # tau de config.yaml -> d09b_criterio_desempate

consultas = pd.read_csv(DATA / "consultas_desarrollo.csv")
relevancias = pd.read_csv(DATA / "relevancias_desarrollo.csv")
ciegas = pd.read_csv(DATA / "consultas_evaluacion.csv")
qrels = qrels_from_judgements(relevancias)

corpus_textos = muestra[CAMPO].tolist()
corpus_ids = muestra["product_id"].tolist()
query_ids = [str(q) for q in consultas["query_id"]]
query_textos = consultas["query_text"].tolist()
# Las 12 ciegas se cargan aquí, junto al resto, porque la sección J las necesita
# codificadas con CADA modelo: si se dejaran para más abajo habría que volver a
# cargar los modelos locales en memoria solo para 12 frases.
ciegas_textos = ciegas["query_text"].tolist()

print(f"corpus   : {len(corpus_textos)} documentos (plantilla {PLANTILLA})")
print(f"consultas: {len(query_textos)} de desarrollo, {len(qrels)} juzgadas")
n_intenciones = ciegas["evaluation_id"].str.split("-").str[1].nunique()
print(f"ciegas   : {len(ciegas_textos)} = {n_intenciones} intenciones x "
      f"{ciegas['query_type'].nunique()} formulaciones")

## B.1 · Registro de modelos

Cada ficha reproduce lo verificado en A.1 más lo que el modelo necesita para codificar. La columna que más decisiones arrastra es **el contrato de entrada**:

| Modelo | Mecanismo del contrato | ¿Entra en el eje con/sin de D10? |
|---|---|---|
| `jina-v3` | **Adaptadores LoRA** (`retrieval.passage` / `retrieval.query`) — pesos distintos, no texto | ✅ Sí, y cuesta ×2 codificaciones |
| `granite-311m-r2` | **Ninguno** — confirmado en la model card de IBM | ❌ No: no hay contrato que retirar |
| `gemini-2` | **Instrucción dentro del prompt** | ✅ Sí (API, barato) |

> 🔎 **Por qué granite no entra en ese eje (P02, cerrado).** Su `config_sentence_transformers.json` declara `"prompts": {"query": "", "document": ""}`: codificar con y sin contrato daría **los mismos vectores**, la Δ sería 0 por construcción, e inventarle un prefijo mediría un modelo que nadie entrenó. Como §3.1 avisa de que *"no basta con citar la documentación del modelo"* —el fichero prueba lo que hace **la librería**, no lo que entrenó **IBM**—, se revisó la model card entera: ni los backends documentados (`sentence-transformers`, Transformers, ONNX, OpenVINO, vLLM, GGUF) ni las secciones *Usage* y *When to Use This Model* traen instrucción alguna, y en el ejemplo de retrieval consultas y documentos se pasan por igual a `model.encode()`. El contrato real es **texto plano simétrico**: `granite` no compitió en desventaja.

> ⚠️ `jina-v3` exige `trust_remote_code=True` —se ejecuta código del repositorio de Jina, y lo hereda quien ejecute el notebook— y su licencia **`cc-by-nc-4.0`** prohíbe el uso comercial, que es justo el escenario de un marketplace. §3.1 obliga a pesar esa clase de restricciones al elegir.

In [ ]:
# 📄 DATOS · ninguno: la ficha técnica de cada modelo candidato
REGISTRO = {
    "jina-v3": {
        "repo": "jinaai/jina-embeddings-v3",
        "ventana": 8192,
        "dim_nativa": 1024,
        "tasks": {"document": "retrieval.passage", "query": "retrieval.query"},
        "trust_remote_code": True,
        "dims": [1024, 768, 512, 256, 128],
        "licencia": "cc-by-nc-4.0",
        # Fuera del análisis de robustez de la sección J: codificar sus 12
        # ciegas obliga a meter 2,3 GB de pesos en memoria, y el barrido ya
        # lo dejó último de los tres. Se declara aquí, junto a la ficha del
        # modelo, para que la exclusión sea auditable y no una celda saltada.
        "ciegas": False,
    },
    "granite-311m-r2": {
        "repo": "ibm-granite/granite-embedding-311m-multilingual-r2",
        "ventana": 32768,
        "dim_nativa": 768,
        "tasks": None,  # prompts declarados como cadena vacía: no hay contrato
        "trust_remote_code": False,
        "dims": [768, 512, 256, 128],
        "licencia": "apache-2.0",
    },
    "gemini-2": {
        "repo": "gemini-embedding-2",
        "ventana": VENTANA_GEMINI,
        "dim_nativa": 3072,
        "api": True,
        "dims": [3072, 1536, 768, 512, 256, 128],
        "licencia": "servicio de terceros",
    },
}


def fabricar(alias):
    """Construye el encoder del alias. Se llama justo antes de codificar y el
    objeto se libera después: dos modelos locales a la vez no caben en 8 GB."""
    ficha = REGISTRO[alias]
    if ficha.get("api"):
        return GeminiEncoder(
            api_key=os.environ.get("GEMINI_API_KEY"),
            model_id=ficha["repo"],
            native_dim=ficha["dim_nativa"],
            window=ficha["ventana"],
        )
    return SentenceTransformerEncoder(
        ficha["repo"],
        window=ficha["ventana"],
        native_dim=ficha["dim_nativa"],
        tasks=ficha["tasks"],
        trust_remote_code=ficha["trust_remote_code"],
        device="cpu",
        token=os.environ.get("HF_TOKEN"),
    )


pd.DataFrame([
    {
        "alias": alias,
        "repo": f["repo"],
        "dim_nativa": f["dim_nativa"],
        "ventana": f["ventana"],
        "tiene_contrato": bool(f.get("api") or f.get("tasks")),
        "licencia": f["licencia"],
    }
    for alias, f in REGISTRO.items()
])

## B.2 · Codificar — ⏱️ **una celda por modelo, ejecutables por separado**

Los tres modelos **no** se codifican en un bucle: cada uno tiene su celda, lanzable por separado y en sesiones distintas. Tres razones, ninguna estética:

1. **RAM.** `jina-v3` ocupa ~2,3 GB de los 7,9 de la máquina. Cada celda construye el modelo, codifica y lo libera con `gc.collect()`: dos modelos vivos a la vez no caben.
2. **Tiempo.** Decenas de minutos por modelo; un bucle único obliga a esperar a los tres para ver el primer número.
3. **Aislamiento de fallos.** Si `jina-v3` revienta por su `trust_remote_code` o Gemini se queda sin cuota, los demás ya están medidos — y con dos de tres sigue habiendo las *"al menos dos configuraciones relevantes"* que pide el enunciado.

### 🔑 Cada celda codifica las DOS variantes

`nativo` (con el contrato de entrada que el modelo declara) y `sin_contrato` (omitiéndolo), juntas a propósito: es lo que hace que **el notebook dé el mismo resultado se ejecute como se ejecute**. La sección C evalúa todo lo que encuentre en `VECTORES`, así que si `sin_contrato` se codificara en la D —donde se analiza—, una ejecución de principio a fin llegaría a C con media tabla y **C.1, C.2, F y G decidirían sobre ella sin avisar**. La D, por tanto, no codifica nada: solo mide la diferencia entre dos ramas que ya existen.

> `granite-311m-r2` no tiene contrato que retirar, así que su segunda llamada no codifica y lo dice por pantalla: se ve que se saltó a propósito y no por olvido (**P02**).

Cada celda deposita sus vectores en `VECTORES`, indexado por `(modelo, contrato)`, y **todo lo que viene detrás —barrido C, contrato D, métrica E, comparación F, regla G— lee ese diccionario y trabaja con lo que encuentre**: las tablas salen con uno, dos o tres modelos.

> 🔁 **Tras reiniciar el kernel** hay que reejecutar las tres celdas, pero con la caché son segundos. Reejecutar tampoco duplica nada: `COSTES` está indexado por `(modelo, contrato, tipo)` y sobrescribe en vez de acumular.

In [ ]:
# 📄 DATOS · 🔬 muestra (documentos) + las 8 de desarrollo + las 12 ciegas
VECTORES = {}   # (alias, contrato) -> {"document", "query", "query_ciegas"}
COSTES = {}     # (alias, contrato, tipo) -> fila de coste. Dict, no lista: así
                # re-ejecutar la celda de un modelo sobrescribe en vez de duplicar.
ERRORES = {}


def codificar(alias, contrato="nativo"):
    """Codifica documentos y consultas de un modelo, y libera la memoria."""
    ficha = REGISTRO[alias]
    encoder = fabricar(alias)
    lotes = 32 if ficha.get("api") else BATCH_LOCAL
    salida = {}
    try:
        # Los corpus del modelo en una sola pasada. `clave` distingue las dos
        # tandas de consultas, que para el encoder son el mismo `kind`: las de
        # desarrollo deciden (tienen juicios) y las ciegas miden robustez (no
        # los tienen). Codificarlas aquí y no en la sección J evita volver a
        # cargar los pesos del modelo para 12 frases.
        corpus = [
            ("document", "document", corpus_textos, CORPUS_ID),
            ("query", "query", query_textos, "consultas_desarrollo"),
        ]
        if ficha.get("ciegas", True):
            corpus.append(
                ("query_ciegas", "query", ciegas_textos, "consultas_evaluacion")
            )
        for clave, kind, textos, corpus_id in corpus:
            resultado = encode_corpus(
                encoder, textos, corpus_id=corpus_id, kind=kind,
                contract=contrato, batch_size=lotes, cache_dir=CACHE,
            )
            salida[clave] = resultado.vectors
            COSTES[(alias, contrato, clave)] = {
                "alias": alias, **resultado.stats.as_row(), "tipo": clave,
            }
    finally:
        # El `finally` importa: si la codificación de consultas falla, el modelo
        # se libera igual y el kernel no se queda con 2,3 GB retenidos.
        del encoder
        gc.collect()
    return salida


def ejecutar(alias, contrato="nativo"):
    """Codifica un modelo dejando el resultado en VECTORES, sin propagar el fallo.

    Cada celda de modelo llama aquí dos veces, una por rama de contrato. Un error
    se registra y se muestra, pero no detiene el notebook: los modelos que sí
    funcionaron siguen siendo medibles."""
    ficha = REGISTRO[alias]
    if contrato == "sin_contrato" and not (ficha.get("api") or ficha.get("tasks")):
        print(f"⏭️  {alias}: no tiene contrato de entrada, el eje no aplica")
        return
    inicio = time.perf_counter()
    try:
        VECTORES[(alias, contrato)] = codificar(alias, contrato)
        ERRORES.pop(f"{alias}[{contrato}]", None)
        print(f"✅ {alias} [{contrato}] listo en {time.perf_counter() - inicio:.1f}s")
    except Exception as error:
        ERRORES[f"{alias}[{contrato}]"] = f"{type(error).__name__}: {error}"
        print(f"⛔ {alias} [{contrato}]: {ERRORES[f'{alias}[{contrato}]'][:250]}")


def estado():
    """Qué hay codificado ahora mismo. Es lo que podrán medir las secciones C-G."""
    filas = [
        {
            "modelo": alias,
            "contrato": contrato,
            "docs": VECTORES[(alias, contrato)]["document"].shape,
            "consultas": VECTORES[(alias, contrato)]["query"].shape,
            "ciegas": VECTORES[(alias, contrato)].get("query_ciegas", np.empty((0, 0))).shape,
        }
        for (alias, contrato) in sorted(VECTORES)
    ]
    return pd.DataFrame(filas) if filas else "Todavía no hay ningún modelo codificado."


print(
    "Listo. Ejecuta las tres celdas siguientes en el orden que prefieras.\n"
    "Cada una codifica su modelo en las DOS ramas de contrato: la sección D las\n"
    "compara, pero no las genera, así que ninguna sección posterior depende del\n"
    "orden en que ejecutes esto."
)

### B.2a · `jina-v3` — ⏱️ el más caro (572M de parámetros)

**Dos codificaciones completas de los 1.500 documentos.** En jina el contrato no es un prefijo de texto sino un **adaptador LoRA**: `retrieval.passage` para documentos y `retrieval.query` para consultas. Retirarlo significa usar otros pesos, así que la variante `sin_contrato` no se puede derivar de la primera — hay que codificar de nuevo. Es el eje más caro de todo D10, y por eso se paga aquí una sola vez.

⚠️ Ejecuta código del repositorio de Jina (`trust_remote_code=True`) y su licencia es **`cc-by-nc-4.0`**: no bloquea el experimento académico, pero sí la recomendación final para un marketplace real. Queda anotado para el informe.

In [ ]:
# 📄 DATOS · 🔬 muestra (1.500 documentos) + 8 consultas de desarrollo. Sin ciegas: ver J
ejecutar("jina-v3")                            # con contrato: adaptador LoRA por tarea
ejecutar("jina-v3", contrato="sin_contrato")   # sin él: mismos textos, otros pesos

### B.2b · `granite-311m-r2` — Apache-2.0, sin código remoto

**Una sola codificación.** La segunda llamada está puesta pero no codifica: granite declara sus dos prompts como cadena vacía, así que `nativo` y `sin_contrato` darían vectores idénticos. Imprime el motivo del salto en vez de gastar otra pasada en una Δ que es 0 por construcción.

La exclusión se registró como **P02** y está **cerrada**: la model card de IBM no documenta instrucción ni prefijo en ningún backend, así que se sostiene en la fuente primaria y no en un JSON de configuración.

In [ ]:
# 📄 DATOS · 🔬 muestra (1.500 documentos) + 8 de desarrollo + 12 ciegas
ejecutar("granite-311m-r2")
# No codifica nada: granite no declara contrato que retirar. La llamada se deja
# puesta para que el salto aparezca en la salida del notebook — así el informe
# muestra que se omitió a propósito, no por olvido (P02, cerrado).
ejecutar("granite-311m-r2", contrato="sin_contrato")

### B.2c · `gemini-2` — por API

**Dos codificaciones, pero baratas.** En Gemini el contrato es texto: una instrucción de tarea antepuesta al contenido. Retirarla no cambia los pesos, solo lo que se envía, y el trabajo lo hace la API — el eje cuesta llamadas, no horas de CPU.

Necesita `GEMINI_API_KEY` en `.env`; sin ella las dos llamadas fallan de forma controlada y el notebook sigue con los modelos locales.

In [ ]:
# 📄 DATOS · 🔬 muestra (1.500 documentos) + 8 de desarrollo + 12 ciegas
ejecutar("gemini-2")                            # con la instrucción de tarea en el prompt
ejecutar("gemini-2", contrato="sin_contrato")   # con el texto desnudo

### B.2d · Qué hay codificado y cuánto ha costado

El coste va **junto** a la calidad, no en una nota al pie: una ventaja de nDCG que se paga con 3× de tiempo de indexación es una decisión distinta a una ventaja gratis. Es el criterio que D09b declaró de antemano.

In [ ]:
# 📄 DATOS · ninguno: resume lo que ya está codificado y lo que costó
display(estado())
if ERRORES:
    display(pd.DataFrame([{"modelo": k, "error": v} for k, v in ERRORES.items()]))
pd.DataFrame(COSTES.values())

## B.3 · Salud de los vectores — antes de creerse ninguna métrica

Un `NaN` o una matriz con filas repetidas producen métricas presentables y falsas. Se comprueban finitud, normas y duplicados antes de creerse ningún número.

**Atención a la columna `normalizado`, que no resultó anecdótica.** `SentenceTransformerEncoder` pide `normalize_embeddings=False` a los dos modelos locales, pero `jina-v3` y `gemini-2` salen con norma exactamente 1 y solo `granite-311m-r2` entrega la salida cruda: ese flag **añade** normalización cuando vale `True`, no **retira** el módulo `Normalize` que jina lleva en su pipeline, ni impide que Gemini devuelva unitarios por API.

Consecuencia: `granite` es el único candidato con el que la sección E puede medir algo.

In [ ]:
# 📄 DATOS · vectores de 🔬 la muestra y de sus consultas, ya en memoria
pd.DataFrame([
    {"alias": alias, "contrato": contrato, "tipo": kind, **vector_health(matriz)}
    for (alias, contrato), por_tipo in VECTORES.items()
    for kind, matriz in por_tipo.items()
])

---

# C · Barrido de dimensión (MRL) — el eje gratis

Los tres candidatos están entrenados con **Matryoshka Representation Learning**: las primeras componentes concentran la mayor parte de la información, así que quedarse con un prefijo del vector reduce la dimensión sin recodificar.

Truncar **obliga a renormalizar** —el prefijo de un vector unitario tiene norma < 1, y sin renormalizar el coseno deja de ser un coseno—; `truncate_dim` lo hace siempre.

Lo que se busca no es el máximo sino **dónde se cae la curva**: si 256 dimensiones pierden menos de 0,02 de nDCG@10 frente a 1.024, el ahorro es de 4× en memoria del motor y en ancho de banda por consulta. Es el compromiso que D09b declaró de antemano.

In [ ]:
# 📄 DATOS · 🔬 muestra (1.500 candidatos) · 8 consultas de desarrollo · 248 juicios
def evaluar(vectores, dim, *, metric="cosine", normalizar=True):
    """Evalúa una configuración concreta sobre las 8 consultas de desarrollo.

    Devuelve el informe y los rankings: guardar los IDs y no solo la métrica es
    lo que permite atribuir errores en NB09 (Regla 3 de experimentación)."""
    docs = truncate_dim(vectores["document"], dim, renormalize=normalizar)
    queries = truncate_dim(vectores["query"], dim, renormalize=normalizar)
    retriever = DenseRetriever(docs, corpus_ids, metric=metric)
    rankings = rank_queries_dense(retriever, query_ids, queries, k=TOP_K)
    return evaluate_rankings(rankings, qrels, k=TOP_K), rankings


costes = pd.DataFrame(COSTES.values())
segundos_por_modelo = (
    costes.query("tipo == 'document'").set_index(["alias", "contrato"])["segundos"]
    if len(costes) else pd.Series(dtype=float)
)

if not VECTORES:
    raise RuntimeError(
        "No hay ningún modelo codificado: ejecuta B.2a, B.2b o B.2c antes de esta celda."
    )

# El barrido recorre TODO lo codificado: cada modelo en sus dos ramas de
# contrato. Filtrar aquí por `nativo` haría que la regla D09b de la sección G
# eligiera al ganador dentro de una sola rama, sin llegar a ver la otra. Y no
# cuesta ninguna codificación extra: truncar y evaluar es el eje gratis de D10.
BARRIDO = []
RANKINGS_DENSOS = {}
for (alias, contrato), vectores in VECTORES.items():
    for dim in REGISTRO[alias]["dims"]:
        informe, rankings = evaluar(vectores, dim)
        RANKINGS_DENSOS[(alias, contrato, dim)] = rankings
        BARRIDO.append({
            "modelo": alias,
            "contrato": contrato,
            # Etiqueta única de la configuración: `modelo` ya no la identifica,
            # porque ahora hay dos filas por modelo y dimensión.
            "sistema": f"{alias} [{contrato}]",
            "dim": dim,
            **informe.summary,
            "bytes_por_vector": dim * 4,
            "segundos": float(segundos_por_modelo.get((alias, contrato), float("nan"))),
        })

barrido = pd.DataFrame(BARRIDO).sort_values("ndcg_at_10", ascending=False)

# Qué ha entrado en el barrido, en voz alta: es la única forma de que quien lea
# el notebook sepa sobre qué se está decidiendo sin auditar el diccionario.
print(f"Evaluadas {len(barrido)} configuraciones sobre {len(query_ids)} consultas:")
for (alias, contrato), grupo in barrido.groupby(["modelo", "contrato"]):
    dims = ", ".join(str(d) for d in sorted(grupo["dim"], reverse=True))
    print(f"  · {alias:<16} [{contrato:<12}]  dims: {dims}")

faltan_ramas = [
    alias for alias, f in REGISTRO.items()
    if (f.get("api") or f.get("tasks")) and (alias, "sin_contrato") not in VECTORES
]
if faltan_ramas:
    print(
        f"\n⚠️  Sin la rama `sin_contrato`: {', '.join(faltan_ramas)}. Tienen contrato de"
        "\n    entrada, así que el barrido está incompleto y D09b decidiría sobre media"
        "\n    tabla. Ejecuta su celda de B.2 entera y vuelve aquí."
    )

barrido

### C.1 · Curva calidad ↔ dimensión

Los mismos números del barrido, en la forma que responde a la pregunta real: **¿dónde se cae la curva?** Si la caída es suave, MRL está funcionando; si hay un escalón, esa dimensión ya no basta para este catálogo.

Tres elementos del gráfico que no son decoración:

- **Color = modelo · trazo = contrato.** Son dos ejes cruzados: fundidos en el color darían cinco tonos sin relación aparente; separados, el ojo agrupa por modelo y compara la continua con la discontinua **dentro** de cada uno. Esa comparación —el mismo modelo consigo mismo— es la que cuantifica la sección D.
- **La banda gris es la tolerancia τ = 0,02 de D09b.** Todo punto que cae dentro es admisible, y entre los admisibles gana el de menor dimensión: **el admisible más a la izquierda**. G lo calcula con `apply_tolerance_rule`; aquí se ve venir.
- **Eje X logarítmico** porque las dimensiones se barren dividiendo por dos: en lineal, 128 y 256 se amontonarían contra el margen y la zona donde se decide el ahorro quedaría ilegible.

La lógica vive en `aurum.graficas`, cubierta por `tests/test_graficas.py`: el notebook declara *qué* quiere ver, no *cómo* se dibuja.

In [ ]:
# 📄 DATOS · el barrido, medido sobre 🔬 la muestra
plot_dimension_curve(
    barrido,
    model_column="modelo",     # el color agrupa por modelo
    dash_column="contrato",    # el trazo separa con/sin contrato dentro de cada uno
    tolerance=TOLERANCIA_D09B,
    subtitle=(
        f"{CORPUS_ID} ({len(corpus_textos)} docs) · {len(query_ids)} consultas · "
        f"la banda gris es la tolerancia τ={TOLERANCIA_D09B} de D09b"
    ),
).show()

### C.2 · Tabla por consulta — la media esconde el caso 33633

NB00 midió que la consulta **33633** (*disfraz halloween talla grande hombre*) tiene **un solo `Exact`** en todo el pool: su Recall@10 solo puede valer 0 o 1, y una media macro sobre 8 consultas se mueve 0,125 según caiga. Reportar solo la media dejaría que esa consulta decidiera el modelo.

In [ ]:
# 📄 DATOS · el barrido sobre 🔬 la muestra, desglosado por las 8 de desarrollo
# Mejor configuración de cada modelo, ya **entre las dos ramas de contrato**:
# si `sin_contrato` gana, es esa la que representa al modelo de aquí en adelante.
mejor_por_modelo = barrido.loc[barrido.groupby("modelo")["ndcg_at_10"].idxmax()]

por_consulta = pd.concat([
    # `fila.contrato` y no "nativo" a mano: si se fijara, se leerían los vectores
    # de la otra rama sin ningún error visible —`(alias, "nativo")` también
    # existe— y la tabla mostraría por consulta un sistema que no es el que ganó.
    evaluar(VECTORES[(fila.modelo, fila.contrato)], int(fila.dim))[0]
    .per_query_frame()
    .assign(sistema=f"{fila.sistema}@{int(fila.dim)}")
    for fila in mejor_por_modelo.itertuples()
])
por_consulta.pivot(index="query_id", columns="sistema", values="ndcg@10")

---

# D · El contrato de entrada (eje "prefijos" de D10)

> 📌 **Esta sección no codifica nada**: las dos variantes de cada modelo se generaron en **B.2**. Aquí solo se comparan, así que corre en segundos.

§3.1 pide elegir cuatro cosas y justificarlas juntas: *"Después elegid **la representación textual, el modelo de embeddings, los prefijos que requiera y la normalización**. No basta con citar la documentación del modelo: la elección debe apoyarse en los resultados de desarrollo y en las restricciones del caso."* El sujeto de "la elección" es la enumeración entera, así que la exigencia se reparte por todo NB02: la representación está congelada en A0 (NB01/NB03), el modelo lo deciden el barrido de C y la regla de G, la normalización se mide en E — y los **prefijos** son esta sección.

Comparar cada modelo consigo mismo, con y sin su contrato, es lo que convierte la cita en evidencia:

- Si retirar el contrato **no cambia nada** (Δ ≈ 0), es que no se estaba aplicando: un fallo de integración disfrazado de resultado.
- Si **cambia**, queda demostrado con datos que el contrato hace algo — y el signo dice si ayuda o estorba en *este* catálogo, que no tiene por qué coincidir con lo que promete la model card.

Qué significa "sin contrato" en cada uno:

| Modelo | `nativo` | `sin_contrato` |
|---|---|---|
| `jina-v3` | Adaptador LoRA `retrieval.passage` / `retrieval.query` | Sin adaptador de tarea — **otros pesos** |
| `gemini-2` | Instrucción de tarea antepuesta al texto | Texto desnudo |
| `granite-311m-r2` | — | **No aplica**: sus dos prompts declarados son cadena vacía |

> ✅ **P02, cerrado.** Esa última exclusión era justo la que §3.1 no admite tal cual: se apoyaba en el `config_sentence_transformers.json` de granite, es decir, en **la documentación del modelo**. Se cerró con la fuente primaria —la model card completa de IBM, incluidas las secciones *Usage* y *When to Use This Model*—, que **no documenta instrucción ni prefijo en ningún backend**: en su ejemplo de retrieval cross-lingual, `input_queries` e `input_passages` van directos a `model.encode()`, a diferencia de `e5-instruct` o los BGE. El contrato real es **texto plano simétrico**, así que granite no compitió en desventaja.

### D.2 · Δ nDCG@10 al retirar el contrato

In [ ]:
# 📄 DATOS · el barrido sobre 🔬 la muestra: las dos ramas de contrato enfrentadas
filas = []
for alias in REGISTRO:
    if (alias, "sin_contrato") not in VECTORES:
        continue
    dim = REGISTRO[alias]["dim_nativa"]
    con, _ = evaluar(VECTORES[(alias, "nativo")], dim)
    sin, _ = evaluar(VECTORES[(alias, "sin_contrato")], dim)
    filas.append({
        "modelo": alias,
        "dim": dim,
        "ndcg_con_contrato": con.summary["ndcg_at_10"],
        "ndcg_sin_contrato": sin.summary["ndcg_at_10"],
        "delta": round(con.summary["ndcg_at_10"] - sin.summary["ndcg_at_10"], 4),
    })

pd.DataFrame(filas) if filas else "Sin modelos con contrato codificados"

---

# E · Normalización y métrica — qué significa el score

§3.2 pide *"conservar la semántica del score nativo"* y §3.1 *"explicar la relación entre la métrica configurada, la normalización y el significado del score"*. Estas dos celdas son esa explicación, medida:

- Con vectores **L2-normalizados**, `cosine` y `dot` dan el **mismo ranking** (el producto escalar de dos unitarios *es* el coseno), y `l2` también, porque `‖a−b‖² = 2 − 2·a·b` es monótona decreciente del producto escalar.
- **Sin normalizar**, `dot` premia los vectores de norma grande y el ranking cambia. Ese es el fallo silencioso que la comprobación caza: si las tres filas normalizadas no coinciden, la normalización no se está aplicando y **todas las métricas del notebook quedan en duda**.

### ⚠️ Cómo leer la tabla: solo un modelo demuestra algo

`jina-v3` y `gemini-2` ya entregan unitarios (B.3), así que su fila *"sin normalizar"* sale idéntica a la normalizada: solo confirma que normalizar dos veces es idempotente. **La evidencia la aporta `granite-311m-r2`**, el único que llega crudo: ahí `cosine` no se mueve —normaliza internamente— mientras `dot` baja y `l2` sube, y las tres dejan de coincidir.

Y llama la atención lo poco que hace falta para romperlo: las normas de granite se desvían milésimas de 1 y ya basta para reordenar resultados y mover el nDCG.

> Si los tres modelos normalizaran en origen, esta comprobación pasaría sin detectar nada. Es el modo en que este tipo de verificación falla en silencio.

In [ ]:
# 📄 DATOS · vectores de 🔬 la muestra, evaluados con las 8 de desarrollo
filas_semantica = []
for (alias, contrato), vectores in VECTORES.items():
    # Solo la rama `nativo`: lo que se demuestra aquí es una propiedad geométrica
    # de los vectores (con norma 1, coseno·dot·l2 ordenan igual), y esa propiedad
    # no depende del contrato con que se generaran. Recorrer las dos ramas
    # duplicaría las filas de la tabla sin añadir ninguna información nueva.
    if contrato != "nativo":
        continue
    dim = REGISTRO[alias]["dim_nativa"]
    for normalizar in (True, False):
        rankings_por_metrica, ndcg_por_metrica = {}, {}
        for metric in ("cosine", "dot", "l2"):
            informe, rankings = evaluar(vectores, dim, metric=metric, normalizar=normalizar)
            rankings_por_metrica[metric] = rankings
            ndcg_por_metrica[metric] = informe.summary["ndcg_at_10"]
        iguales = (
            rankings_por_metrica["cosine"] == rankings_por_metrica["dot"] == rankings_por_metrica["l2"]
        )
        # `mismo_ranking` se guarda en la fila, no solo se imprime: es la
        # conclusión de la sección y tiene que sobrevivir a un reinicio.
        filas_semantica += [
            {"modelo": alias, "dim": dim, "normalizado": normalizar,
             "metrica": metric, "ndcg_at_10": valor, "mismo_ranking": iguales}
            for metric, valor in ndcg_por_metrica.items()
        ]
        print(f"{alias} · normalizado={normalizar}: ¿mismo ranking en las 3 métricas? {iguales}")

semantica_score = pd.DataFrame(filas_semantica)
semantica_score.pivot(index=["modelo", "normalizado"], columns="metrica", values="ndcg_at_10")

---

# F · Contra el baseline léxico — el requisito del enunciado §3.1

> *"El trabajo debe comparar el sistema denso con, al menos, un baseline léxico o exacto."*

NB01 dejó ese baseline en `artifacts/baseline_lexico.json`. Aquí se recupera **sobre el mismo corpus** (la muestra de 1.500), con las mismas 8 consultas, el mismo `k`, los mismos qrels y el mismo contrato de relevancia: sin esa igualdad no se compararían métodos sino entornos (Regla 2).

La pregunta no es *"¿gana el denso?"* sino **"¿cuánto gana y a cambio de qué coste?"**: BM25 se construye en segundos sobre CPU, sin modelo ni base vectorial. Si la mejora fuera marginal, el argumento de negocio para montar esta infraestructura sería flojo — y decirlo con un número es mejor informe que esconderlo.

In [ ]:
# 📄 DATOS · artifacts/baseline_lexico.json, rama "muestra" — el mismo corpus 🔬
import json

baseline = json.loads(
    (Path("..") / "artifacts" / "baseline_lexico.json").read_text(encoding="utf-8")
)
lexico_muestra = baseline["muestra"]["metricas"]

# `modelo` y `contrato` viajan hasta aquí aunque no se muestren: F.1 los necesita
# para volver a buscar los vectores del ganador en VECTORES. `sistema` es solo
# la etiqueta legible.
comparativa = pd.concat([
    pd.DataFrame([
        {"sistema": nombre, "familia": "léxico", "modelo": None, "contrato": None,
         "dim": None, **metricas}
        for nombre, metricas in lexico_muestra.items()
    ]),
    mejor_por_modelo.assign(familia="denso")[
        ["sistema", "familia", "modelo", "contrato", "dim",
         "precision_at_10", "recall_at_10", "mrr_at_10", "ndcg_at_10"]
    ],
]).sort_values("ndcg_at_10", ascending=False).reset_index(drop=True)

mejor_lexico = max(m["ndcg_at_10"] for m in lexico_muestra.values())
comparativa["delta_vs_mejor_lexico"] = (comparativa["ndcg_at_10"] - mejor_lexico).round(4)
comparativa

In [ ]:
# 📄 DATOS · denso y léxico, ambos sobre 🔬 la muestra
# La misma comparativa de arriba en barras agrupadas: las cuatro métricas en una
# escala 0-1 común, que es la forma en que §3.1 pide leer denso frente a léxico.
METRICAS = ["precision_at_10", "recall_at_10", "mrr_at_10", "ndcg_at_10"]


def etiqueta(fila):
    """El denso lleva su dimensión en el nombre: `granite-311m-r2@256` y `@768`
    son sistemas distintos y la leyenda tiene que poder distinguirlos."""
    if fila["familia"] == "léxico":
        return fila["sistema"]
    return f"{fila['sistema']}@{int(fila['dim'])}"


sistemas = {
    etiqueta(fila): {metrica: float(fila[metrica]) for metrica in METRICAS}
    for _, fila in comparativa.iterrows()
}

plot_metric_comparison(
    sistemas,
    title="Denso frente al baseline léxico de NB01",
    subtitle=(
        f"{CORPUS_ID} ({len(corpus_textos)} docs) · {len(query_ids)} consultas · k={TOP_K} "
        "· mismo corpus, mismos qrels, mismo contrato de relevancia"
    ),
).show()

### F.1 · Dónde gana cada familia, consulta a consulta

El agregado dice quién gana; esta tabla dice **por qué**. Lo interesante son las consultas donde el denso mejora mucho (vocabulario distinto al del catálogo) y las que empeora (el léxico acierta por coincidencia literal y el denso trae vecinos semánticamente próximos pero comercialmente distintos). Los dos casos alimentan la atribución de errores de NB09.

In [ ]:
# 📄 DATOS · 🔬 muestra · las 8 de desarrollo, una a una
mejor_denso = comparativa.query("familia == 'denso'").iloc[0]
nombre_lexico = max(lexico_muestra, key=lambda n: lexico_muestra[n]["ndcg_at_10"])
etiqueta_densa = f"{mejor_denso['sistema']}@{int(mejor_denso['dim'])}"

ndcg_lexico = {
    str(fila["query_id"]): fila["ndcg@10"]
    for fila in baseline["muestra"]["por_consulta"][nombre_lexico]
}
# La clave de VECTORES es (modelo, contrato). `sistema` es la etiqueta legible
# —"jina-v3 [sin_contrato]"— y no sirve para buscar aquí.
informe_denso, _ = evaluar(
    VECTORES[(mejor_denso["modelo"], mejor_denso["contrato"])], int(mejor_denso["dim"])
)

frente_a_frente = (
    informe_denso.per_query_frame()
    .assign(
        query_id=lambda d: d["query_id"].astype(str),
        **{nombre_lexico: lambda d: d["query_id"].map(ndcg_lexico)},
    )
    .rename(columns={"ndcg@10": etiqueta_densa})
    [["query_id", nombre_lexico, etiqueta_densa]]
    .merge(consultas.assign(query_id=consultas["query_id"].astype(str)), on="query_id")
)
frente_a_frente["delta"] = (
    frente_a_frente[etiqueta_densa] - frente_a_frente[nombre_lexico]
).round(4)
frente_a_frente[["query_id", "query_text", nombre_lexico, etiqueta_densa, "delta"]].sort_values("delta")

---

# G · R02 · Aplicar D09b y dejar el artefacto

**D09b se fijó en `config/config.yaml` antes de codificar nada.** Aplicarla como función y no a ojo es lo que hace verificable esa afirmación: el ganador sale de `apply_tolerance_rule`, que es determinista y está cubierta por tests.

```yaml
d09b_criterio_desempate:
  forma: mas_barata_dentro_de_tolerancia
  metrica_primaria: ndcg_at_10
  tolerancia_tau: 0.02
```

1. `B` = mejor nDCG@10 de toda la tabla.
2. Admisibles: las que están a menos de 0,02 de `B` — con 8 consultas, una diferencia menor no distingue dos sistemas.
3. Entre las admisibles gana **la de menor dimensión**; a igualdad, mayor nDCG; después, menor tiempo de codificación.

> ⚖️ **La celda produce la ordenación; R02 se ratifica sobre ella.** Si el resultado sorprende, el sitio para discutirlo es el criterio, no la tabla: cambiar la regla después de ver los números es exactamente lo que el enunciado penaliza.

In [ ]:
# 📄 DATOS · el barrido sobre 🔬 la muestra; la regla no vuelve a leer nada
ordenadas = apply_tolerance_rule(
    barrido, metrica="ndcg_at_10", tolerancia=TOLERANCIA_D09B,
    coste="dim", desempates=("segundos",),
)
# `contrato` en las columnas mostradas: sin él, dos configuraciones distintas
# del mismo modelo y dimensión aparecen como filas idénticas y no hay forma de
# saber cuál ha ganado.
ordenadas[["posicion_regla", "modelo", "contrato", "dim", "ndcg_at_10", "recall_at_10",
           "mrr_at_10", "bytes_por_vector", "segundos", "admisible"]]

In [ ]:
# 📄 DATOS · escribe artifacts/comparativa_modelos.json y .md — todo de 🔬 la muestra
def registros(frame):
    """Filas como tipos JSON nativos: `to_dict` dejaría escalares de numpy."""
    return json.loads(frame.to_json(orient="records"))


artefacto = {
    "configuracion": {
        "corpus": CORPUS_ID,
        "n_docs": len(corpus_textos),
        "plantilla": PLANTILLA,
        "campo": CAMPO,
        "top_k": TOP_K,
        "relevancia": {"E": 3, "S": 2, "C": 1, "I": 0},
        "d09b": {"metrica": "ndcg_at_10", "tolerancia": TOLERANCIA_D09B, "coste": "dim"},
    },
    "modelos": {alias: {k: v for k, v in f.items() if k != "tasks"} for alias, f in REGISTRO.items()},
    "errores_de_codificacion": ERRORES,
    "costes_de_codificacion": list(COSTES.values()),
    "modelos_codificados": [f"{a}[{c}]" for a, c in sorted(VECTORES)],
    "barrido": registros(barrido),
    "regla_d09b": registros(ordenadas),
    "comparativa_con_lexico": registros(comparativa),
    # §3.1 pide explicar la relación entre métrica, normalización y score. Sin
    # esto, esa evidencia vivía solo en la salida de una celda.
    "semantica_del_score": registros(semantica_score),
    # La clave lleva el contrato: el barrido tiene ahora dos rankings por modelo
    # y dimensión, y sin él uno sobrescribiría al otro en silencio.
    "rankings": {
        f"{alias}[{contrato}]@{dim}": r
        for (alias, contrato, dim), r in RANKINGS_DENSOS.items()
    },
}

destino = Path("..") / "artifacts" / "comparativa_modelos.json"
destino.write_text(json.dumps(artefacto, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

markdown = Path("..") / "artifacts" / "comparativa_modelos.md"
markdown.write_text(
    "# Comparativa de modelos (NB02)\n\n"
    f"Corpus: `{CORPUS_ID}` ({len(corpus_textos)} docs) · plantilla `{PLANTILLA}` · k={TOP_K}\n\n"
    "## Barrido modelo x contrato x dimension\n\n" + barrido.to_markdown(index=False) + "\n\n"
    "## Regla D09b aplicada\n\n" + ordenadas.to_markdown(index=False) + "\n\n"
    "## Denso frente al baseline lexico de NB01\n\n" + comparativa.to_markdown(index=False) + "\n",
    encoding="utf-8",
)
print(f"Escrito {destino.name} ({destino.stat().st_size / 1024:.1f} KB) y {markdown.name}")

---

# H · El contrato no aporta lo mismo en todas las dimensiones

La sección D midió la Δ del contrato **solo en la dimensión nativa**, y con ese único punto la conclusión parecía limpia: retirarlo mejora en los dos modelos que lo tienen. El barrido completo dice algo más incómodo.

**La Δ cambia de signo al truncar.** En `gemini-2`:

| dim | `nativo` | `sin_contrato` | Δ |
|---:|---:|---:|---:|
| 3072 | 0,7509 | 0,7765 | **−0,0256** |
| 1536 | 0,7496 | 0,7750 | −0,0254 |
| 768 | 0,7478 | 0,7718 | −0,0240 |
| 512 | 0,7110 | 0,7410 | −0,0300 |
| 256 | 0,7101 | 0,6843 | **+0,0258** |
| 128 | 0,6648 | 0,5575 | **+0,1073** |

Por encima de 512 el contrato estorba. Por debajo, ayuda — y a 128 la diferencia es de **0,107**, cinco veces la tolerancia de D09b. El cruce está entre 512 y 256.

### Una hipótesis, no una conclusión

La instrucción de tarea es **texto idéntico en los 1.500 documentos**, y en un modelo con MRL las primeras componentes concentran la estructura más gruesa y compartida del corpus, que es donde esa señal común pesa más.

- **A dimensión completa** ese prefijo compartido es lastre: ocupa norma sin distinguir un producto de otro, así que quitarlo mejora.
- **Al truncar fuerte** quedan casi solo esas componentes, y ahí la instrucción funciona como condicionamiento de tarea que sí orienta la búsqueda.

No está comprobado —habría que mirar la energía por componente en ambas variantes—, así que queda como hipótesis explícita.

### Consecuencia práctica

**"Sin contrato es mejor" no es incondicional: vale a partir de 512.** El ganador de D09b (`gemini-2 [sin_contrato] @768`) cae con holgura en esa zona, así que **R02 no se ve afectada** — pero la condición hay que arrastrarla: si la memoria del índice empujara a bajar de dimensión (15.000 productos son 46 MB a 768 y 15 MB a 256), la decisión sobre el contrato habría que **revisarla, no heredarla**. Medir ese eje en un solo punto habría dejado la trampa puesta.

In [ ]:
# 📄 DATOS · el barrido sobre 🔬 la muestra, dimensión a dimensión
from aurum.graficas import plot_contract_delta

plot_contract_delta(
    barrido,
    tolerance=TOLERANCIA_D09B,
    subtitle=(
        f"{CORPUS_ID} ({len(corpus_textos)} docs) · {len(query_ids)} consultas · "
        f"dentro de la banda gris (±{TOLERANCIA_D09B}) la diferencia no se distingue"
    ),
).show()

# El cruce, en números: dónde deja de convenir retirar el contrato.
cruce = (
    barrido.pivot_table(index=["modelo", "dim"], columns="contrato", values="ndcg_at_10")
    .dropna(subset=["nativo", "sin_contrato"])
    .assign(delta=lambda d: (d["nativo"] - d["sin_contrato"]).round(4))
    .assign(gana=lambda d: d["delta"].apply(
        lambda x: "nativo" if x > TOLERANCIA_D09B
        else ("sin_contrato" if x < -TOLERANCIA_D09B else "indistinguible")
    ))
    .reset_index()
    .sort_values(["modelo", "dim"], ascending=[True, False])
)
cruce[["modelo", "dim", "nativo", "sin_contrato", "delta", "gana"]]

---

# I · P01 · El ganador sobre el catálogo completo

Todo lo anterior está medido sobre **1.500 documentos**. El enunciado (§6, *Condiciones de comparabilidad*) dice que *"el catálogo completo es el recorrido evaluado; la muestra sirve para desarrollar y depurar"*, y esta sección cierra esa distancia para el ganador de D09b.

### Por qué esto no es un detalle

NB01 ya midió lo que pasa al multiplicar por diez los candidatos, con **los mismos juicios de relevancia**:

| baseline | muestra (1.500) | completo (15.000) | caída |
|---|---:|---:|---:|
| BM25 | 0,6512 | 0,5088 | **−0,1424** |
| TF-IDF | 0,5654 | 0,4129 | −0,1525 |

Los qrels no cambian: aparecen 13.500 productos más compitiendo por las 10 posiciones que, al no estar juzgados, puntúan 0 (D04). Un sistema que los sube se desploma; uno que mantiene arriba los juzgados aguanta. **Es un test de precisión bajo distracción**, y no hay forma de aprobarlo desde la muestra.

### Qué se ejecuta aquí, y qué no

Solo se codifica el **ganador de D09b**, leído de `ordenadas` y no escrito a mano: si la regla cambiara de ganador, la sección lo sigue. El motivo es de coste, medido y no estimado a ojo:

| configuración | 1.500 docs | 15.000 (×10) |
|---|---:|---:|
| `gemini-2` | ~50 s | **~8 min** ✅ |
| `jina-v3` | ~5.750 s | ~16 h ❌ |
| `granite-311m-r2` | ~17.490 s | ~49 h ❌ |

La celda calcula esa extrapolación y **se niega a lanzar** cualquier codificación por encima de `LIMITE_HORAS`: con un modelo local imprimiría el coste y saltaría el paso en vez de dejar el kernel bloqueado media semana.

Dos cosas abaratan la prueba: **las consultas no se recodifican** —su caché es independiente del corpus (`corpus_id="consultas_desarrollo"`), así que los 8 vectores ya están en disco— y **las tres dimensiones admisibles salen de la misma codificación**, porque truncar es gratis y 768, 1.536 y 3.072 se evalúan sin ninguna llamada extra.

### Qué responde y qué no

Responde a la pregunta del enunciado —**¿el denso sigue batiendo al léxico con el catálogo de verdad?**— pero no al orden **entre modelos densos** a esa escala: para eso habría que pagar las 65 horas de jina y granite. Queda como límite explícito: con `gemini-2` sacando 0,12 sobre BM25 y 0,24 sobre jina en la muestra, el riesgo de que el orden se invierta es bajo, pero *bajo* no es *cero*.

In [ ]:
# 📄 DATOS · 📚 catalogo_productos.csv (15.000) — AQUÍ CAMBIA EL CORPUS: hasta la
#            sección H todo iba sobre la muestra de 1.500
LIMITE_HORAS = 1.0            # por encima de esto la celda no lanza la codificación
CORPUS_COMPLETO = "catalogo_productos"

# El ganador se lee de la regla, no se escribe a mano: si D09b cambiara de
# resultado, esta sección lo sigue sin tocar una línea.
ganador = ordenadas.iloc[0]
ALIAS_G = ganador["modelo"]
CONTRATO_G = ganador["contrato"]
DIM_G = int(ganador["dim"])
ETIQUETA_G = f"{ALIAS_G} [{CONTRATO_G}]@{DIM_G}"

textos_completo = completo[CAMPO].tolist()
ids_completo = completo["product_id"].tolist()

# Extrapolación lineal desde el coste ya medido sobre la muestra. Es fiable
# porque codificar es proporcional al número de documentos: mismo modelo, mismo
# lote, mismo hardware.
segundos_muestra = float(
    costes.query(
        "alias == @ALIAS_G and contrato == @CONTRATO_G and tipo == 'document'"
    )["segundos"].iloc[0]
)
horas = segundos_muestra * len(textos_completo) / len(corpus_textos) / 3600

print(f"Ganador de D09b : {ETIQUETA_G}")
print(f"Corpus completo : {len(textos_completo)} documentos")
print(f"Coste estimado  : ~{horas:.2f} h  ({segundos_muestra:.0f}s para {len(corpus_textos)} docs)")

if horas > LIMITE_HORAS:
    vectores_completo = None
    print(
        f"\n⏭️  Por encima del límite de {LIMITE_HORAS} h: no se lanza.\n"
        "    P01 queda abierto para esta configuración. Sube LIMITE_HORAS si\n"
        "    quieres pagarlo, pero hazlo sabiendo cuántas horas son."
    )
else:
    encoder = fabricar(ALIAS_G)
    try:
        resultado = encode_corpus(
            encoder, textos_completo, corpus_id=CORPUS_COMPLETO,
            kind="document", contract=CONTRATO_G,
            batch_size=32, cache_dir=CACHE,
        )
    finally:
        del encoder
        gc.collect()
    vectores_completo = resultado.vectors
    origen = "desde caché" if resultado.stats.desde_cache else f"{resultado.stats.segundos:.0f}s"
    print(f"\n✅ {vectores_completo.shape[0]} vectores de {vectores_completo.shape[1]} dims ({origen})")

In [ ]:
# 📄 DATOS · 📚 catalogo_productos.csv (15.000) · 8 de desarrollo · baseline_lexico.json
#            rama "completo" — mismo corpus en los dos lados de la comparación
if vectores_completo is None:
    print("P01 sin cerrar: la celda anterior no codificó el catálogo completo.")
else:
    # Las consultas NO se recodifican: su caché es independiente del corpus.
    vectores_query = VECTORES[(ALIAS_G, CONTRATO_G)]["query"]

    def evaluar_completo(dim):
        """Igual que `evaluar`, pero contra los 15.000 IDs del catálogo entero.

        No se reutiliza `evaluar` porque aquella cerró sobre `corpus_ids`, que
        son los 1.500 de la muestra: pasarle estos vectores devolvería IDs
        equivocados sin dar ningún error."""
        docs = truncate_dim(vectores_completo, dim)
        queries = truncate_dim(vectores_query, dim)
        retriever = DenseRetriever(docs, ids_completo, metric="cosine")
        rankings = rank_queries_dense(retriever, query_ids, queries, k=TOP_K)
        return evaluate_rankings(rankings, qrels, k=TOP_K), rankings

    lexico_completo = baseline["completo"]["metricas"]
    ndcg_muestra = {n: m["ndcg_at_10"] for n, m in lexico_muestra.items()}

    filas = []
    RANKINGS_COMPLETO = {}
    # Las tres dimensiones admisibles de D09b salen de la misma codificación:
    # truncar es gratis, así que verlas todas no cuesta ninguna llamada extra.
    for dim in sorted(ordenadas.query("admisible")["dim"].unique(), reverse=True):
        informe, rankings = evaluar_completo(int(dim))
        RANKINGS_COMPLETO[int(dim)] = rankings
        etiqueta = f"{ALIAS_G} [{CONTRATO_G}]@{int(dim)}"
        ndcg_muestra[etiqueta] = float(
            barrido.query(
                "modelo == @ALIAS_G and contrato == @CONTRATO_G and dim == @dim"
            )["ndcg_at_10"].iloc[0]
        )
        filas.append({"sistema": etiqueta, "familia": "denso", **informe.summary})

    filas += [
        {"sistema": nombre, "familia": "léxico", **metricas}
        for nombre, metricas in lexico_completo.items()
    ]

    completo_vs_lexico = (
        pd.DataFrame(filas).sort_values("ndcg_at_10", ascending=False).reset_index(drop=True)
    )
    completo_vs_lexico["muestra_1500"] = completo_vs_lexico["sistema"].map(ndcg_muestra)
    completo_vs_lexico["caida_al_escalar"] = (
        completo_vs_lexico["ndcg_at_10"] - completo_vs_lexico["muestra_1500"]
    ).round(4)

    mejor_lexico_completo = max(m["ndcg_at_10"] for m in lexico_completo.values())
    ventaja = (
        completo_vs_lexico.query("familia == 'denso'").iloc[0]["ndcg_at_10"]
        - mejor_lexico_completo
    )
    print(
        f"Ventaja del mejor denso sobre el mejor léxico:\n"
        f"  muestra  (1.500) : {barrido.iloc[0]['ndcg_at_10'] - max(ndcg_muestra[n] for n in lexico_muestra):+.4f}\n"
        f"  completo (15.000): {ventaja:+.4f}"
    )
    display(
        completo_vs_lexico[
            ["sistema", "familia", "muestra_1500", "ndcg_at_10",
             "caida_al_escalar", "recall_at_10", "mrr_at_10", "precision_at_10"]
        ]
    )

    plot_metric_comparison(
        {
            fila["sistema"]: {m: float(fila[m]) for m in METRICAS}
            for _, fila in completo_vs_lexico.iterrows()
        },
        title="Denso frente al léxico — catálogo completo",
        subtitle=(
            f"{CORPUS_COMPLETO} ({len(textos_completo)} docs) · {len(query_ids)} consultas "
            f"· k={TOP_K} · mismos qrels que sobre la muestra"
        ),
    ).show()

---

# J · Robustez entre formulaciones — el slice que el barrido no ve

Las secciones A–I eligen el modelo con **8 consultas de desarrollo**, el único conjunto con juicios pero con un punto ciego: un modelo puede ganar por afinidad léxica con esas ocho. Queda sin responder si devuelve **los mismos productos cuando la misma intención se escribe de otra manera**, que es lo que pasa en producción.

Las 12 ciegas son **4 intenciones × 3 formulaciones** y no necesitan juicios para eso: el Jaccard@10 entre las tres formas de pedir lo mismo mide la estabilidad directamente.

```
EVAL-100455-direct    "taladro 24v batería"
EVAL-100455-context   "taladro sin cable de 24 voltios que venga con su batería"
EVAL-100455-semantic  "quiero una herramienta inalámbrica potente para perforar sin depender de un enchufe"
```

> 🔍 **Por qué llega después de R02.** La consistencia estaba medida para los baselines (NB01) y para las plantillas (NB03), pero no **entre modelos**, que es una de las patas del criterio *"media y por slices"*. Se añade como comprobación de una decisión ya tomada: si el ganador por nDCG resultara el más inestable, R02 se revisa en vez de esconderse. Se mide sobre la muestra porque es donde están codificados los modelos que compiten (Regla 2); la cifra del ganador sobre el catálogo completo está en NB03.

> ⏭️ **`jina-v3` no entra aquí**, y queda declarado en su ficha del registro en vez de saltado en una celda: codificar sus 12 ciegas obliga a cargar 572M de parámetros —2,3 GB de los 7,9 de la máquina— y el barrido ya lo dejó **último de los tres**, por debajo incluso del baseline léxico. La pregunta es cuál de los dos que de verdad compiten aguanta mejor un cambio de formulación, y añadir al descartado no cambiaría esa respuesta, solo el tiempo de ejecución. Es una limitación del equipo, asumida a la vista de un orden ya medido.

In [ ]:
# 📄 DATOS · 🔬 muestra (1.500 candidatos) · consultas_evaluacion.csv (12 ciegas, sin juicios)
# Cada modelo se mide en SU mejor configuración —la que ganó su rama en el
# barrido—, no todos a la misma dimensión: forzar un valor común mediría el
# efecto de la dimensión, no el del modelo.
consistencia_modelos = []
for _, fila in mejor_por_modelo.iterrows():
    vectores = VECTORES[(fila["modelo"], fila["contrato"])]
    if "query_ciegas" not in vectores:
        motivo = (
            "excluido a propósito (REGISTRO: ciegas=False)"
            if not REGISTRO[fila["modelo"]].get("ciegas", True)
            else "sin las ciegas codificadas: re-ejecuta su celda B.2"
        )
        print(f"⏭️  {fila['sistema']}: {motivo}")
        continue
    dim = int(fila["dim"])
    retriever = DenseRetriever(
        truncate_dim(vectores["document"], dim), corpus_ids, metric="cosine"
    )
    rankings = rank_queries_dense(
        retriever, ciegas["evaluation_id"].tolist(),
        truncate_dim(vectores["query_ciegas"], dim), k=TOP_K,
    )
    tabla = formulation_consistency(rankings, k=TOP_K)
    columnas = [c for c in tabla.columns if c.startswith("jaccard_")]
    consistencia_modelos.append({
        "sistema": fila["sistema"],
        "dim": dim,
        "ndcg_at_10": fila["ndcg_at_10"],
        **{c: round(float(tabla[c].mean()), 4) for c in columnas},
        "jaccard_medio": round(float(tabla[columnas].to_numpy().mean()), 4),
    })

consistencia_modelos = (
    pd.DataFrame(consistencia_modelos)
    .sort_values("jaccard_medio", ascending=False)
    .reset_index(drop=True)
)
consistencia_modelos

### Cómo se lee esta tabla

La columna que importa no es solo `jaccard_medio` sino **el par `direct` ↔ `semantic`**: las dos formulaciones más lejanas, una en palabras del catálogo y otra en las de un cliente que describe lo que necesita. Ahí se cae un sistema que en realidad depende del vocabulario.

Dos lecturas posibles y qué significa cada una:

| Si… | Entonces |
|---|---|
| el orden coincide con el de nDCG@10 | la ventaja del ganador no era afinidad con las 8 consultas: generaliza |
| un modelo gana en nDCG pero pierde aquí | su ventaja **es sobreajuste** a la superficie léxica del conjunto de desarrollo, y R02 hay que revisarla |

> ⚠️ **Cuánto pesa esto.** Son 4 intenciones, no 400: sirve para detectar un derrumbe, no para separar dos modelos que queden cerca. Un Jaccard de 0,50 frente a 0,55 no distingue nada; uno de 0,50 frente a 0,05 sí.

---

# K · El coste del camino online — indexar una vez, buscar siempre

Todo lo medido hasta aquí es **coste de indexación**: un lote grande, una vez. Falta la otra mitad, la que se paga en cada búsqueda y para siempre.

**Los números de B.2d no sirven para esto**: allí las 8 consultas se codificaron en un solo lote (0,43 s las ocho con `gemini-2`), y un lote amortiza el viaje de red entre ellas, que es justo lo que domina el camino online. Una búsqueda real codifica **una** consulta y paga el round-trip entero.

Hay además una asimetría entre los dos regímenes que el barrido no ve, porque no es una propiedad del vector:

| | Modelo local | Modelo por API |
|---|---|---|
| Indexar | horas de CPU, 0 $ | minutos, $ por token |
| Buscar | ms de CPU, 0 $ | ms de red, $ por token |
| Si se cae | no se cae: es un fichero | el buscador entero se queda sin codificar consultas |

## K.1 · Latencia de **una** consulta — ⏱️ una celda por modelo

Mismo patrón que B.2: cada modelo en su celda, porque los locales cargan pesos en memoria (`jina-v3` ~2,3 GB) y no caben dos a la vez. `gemini-2` es barato —son llamadas a la API— y los dos locales son opcionales: sirven de referencia, no deciden nada.

El calentamiento no se contabiliza: la primera llamada paga el TLS o la reserva de memoria, y eso no se repite por consulta.

In [ ]:
# 📄 DATOS · consultas_desarrollo.csv (8), aquí solo como textos que codificar
LATENCIAS = {}


def latencia(alias, contrato="nativo", repeticiones=20):
    """Mide y libera. Devuelve None si el modelo no está disponible."""
    try:
        encoder = fabricar(alias)
    except Exception as error:
        print(f"⛔ {alias}: {type(error).__name__} — {error}")
        return None
    try:
        informe = measure_encode_latency(
            encoder, query_textos, kind="query", contract=contrato,
            repeticiones=repeticiones, calentamiento=2,
        )
    finally:
        del encoder
        gc.collect()
    LATENCIAS[(alias, contrato)] = informe
    print(f"✅ {alias} [{contrato}]: p50 {informe['ms_p50']} ms · p95 {informe['ms_p95']} ms")
    return informe


# El ganador de R02, en la rama de contrato que eligió la regla.
latencia("gemini-2", "sin_contrato")

### K.1b · Los dos locales — ⏱️ **opcional**, carga los pesos (1-2 min cada uno)

No cambian ninguna decisión: R02 ya está tomada. Se miden porque son el contrafactual del régimen de coste — *"¿cuánto costaría no depender de una API?"*— y esa cifra es parte del argumento del informe, no un adorno.

In [ ]:
# 📄 DATOS · consultas_desarrollo.csv (8), aquí solo como textos que codificar
latencia("granite-311m-r2")   # Apache-2.0, el más ligero de los dos
# latencia("jina-v3")        # descoméntalo si hay RAM libre: 572M de parámetros

In [ ]:
# 📄 DATOS · ninguno: recoge las latencias ya medidas
if LATENCIAS:
    display(pd.DataFrame(LATENCIAS.values())[
        ["modelo", "contrato", "n_llamadas", "ms_p50", "ms_p95", "ms_min", "ms_max"]
    ])
else:
    print("Sin medidas: ejecuta al menos una celda de K.1.")

## K.2 · Cuánto cuesta en dinero

`gemini-embedding-2` cobra **0,20 $ por millón de tokens** de entrada en la tarifa estándar (0,10 $ con la Batch API, que sirve para indexar pero no para responder a un usuario). El precio es un dato declarado del proveedor, no calculado aquí: puede cambiar, y esconderlo dentro de una celda lo volvería invisible.

> ⚠️ **La cuenta de tokens es una aproximación.** Gemini no publica su tokenizador, así que se usa el de `jina-v3` sobre el mismo texto (sección A): dos vocabularios no parten el texto igual y el total tiene un margen de decenas por ciento. Da igual para lo que se decide aquí — la conclusión son céntimos, y lo seguiría siendo con el doble o la mitad.

**Unidades de las dos tablas**, que mezclan dólares, consultas y ratios:

| Columna | Unidad | Qué es |
|---|---|---|
| `indexacion_completa_usd` | **USD** | codificar las 15.000 fichas, una vez |
| `por_consulta_usd` | **USD** | codificar **una** consulta de usuario |
| `por_1000_consultas_usd` | **USD** | lo mismo × 1.000, para que la cifra se lea |
| `consultas_equivalentes_a_reindexar` | **consultas** | cuántas búsquedas cuestan lo que una reindexación completa |
| `consultas_mes` | **consultas/mes** | volumen supuesto, no medido |
| `gasto_mes_usd` | **USD/mes** | lo que costaría ese volumen |
| `veces_el_coste_de_indexar` | **ratio** (adimensional) | ese gasto mensual dividido por el de indexar |

USD es la moneda en la que factura el proveedor. La conversión a euros no se hace a propósito: el tipo de cambio del día metería ruido en una cifra que ya es una estimación.

In [ ]:
# 📄 DATOS · 📚 tokens de catalogo_productos.csv (15.000, medidos en A.4)
#            + consultas_desarrollo.csv (8) y consultas_evaluacion.csv (12)
# `longitudes` viene de A.4: tokens de cada ficha del catálogo completo con el
# tokenizador de referencia. `tokens_por_consulta` sale de las consultas reales,
# no de una estimación: las 8 de desarrollo más las 12 ciegas.
PRECIO_POR_MILLON = 0.20   # $/1M tokens — config.yaml -> nb02_modelo.coste_api

tokens_consulta = token_lengths(
    query_textos + ciegas_textos, tokenizadores[alias_referencia][0]
)

coste = api_cost_report(
    tokens_indexacion=float(longitudes.sum()),
    tokens_por_consulta=float(tokens_consulta.mean()),
    precio_por_millon=PRECIO_POR_MILLON,
)

print(f"Catálogo completo : {longitudes.sum():>10,.0f} tokens")
print(f"Consulta media    : {tokens_consulta.mean():>10,.1f} tokens")
pd.DataFrame([coste])

In [ ]:
# 📄 DATOS · ninguno: aritmética sobre el informe de coste de la celda anterior
# El gasto por consulta suelta es demasiado pequeño para decir nada. Lo que
# ordena las dos partidas es a qué volumen de búsquedas el coste de buscar
# alcanza al de indexar el catálogo entero.
pd.DataFrame([
    {
        "consultas_mes": volumen,
        "gasto_mes_usd": round(coste["por_consulta_usd"] * volumen, 2),
        "veces_el_coste_de_indexar": round(
            coste["por_consulta_usd"] * volumen / coste["indexacion_completa_usd"], 2
        ),
    }
    for volumen in (10_000, 100_000, 1_000_000, 10_000_000)
])

### Qué sale de la máquina, y en qué momento

El mismo eje que el coste, mirado como exposición en vez de como factura. No es lo mismo lo que se envía al indexar que lo que se envía al buscar, y el enunciado pide pesar la dependencia del proveedor:

| Momento | Qué viaja a Google | Cuántas veces |
|---|---|---|
| Indexación | el texto de las 15.000 fichas del catálogo | una vez, y otra por cada reindexado |
| Búsqueda | **la consulta que ha escrito el usuario** | una por búsqueda, para siempre |
| Altas y actualizaciones (NB08) | el texto de los productos nuevos o modificados | una por evento |

La ficha de producto es información pública: un catálogo de comercio electrónico existe para ser visto. **La consulta no.** Es lo que una persona concreta estaba buscando en un momento concreto, y es el único dato del sistema que no era público antes de entrar en él. La dependencia de la API se paga ahí, no en el catálogo.

> ⚖️ **Esto es evidencia, no una decisión.** Qué se concluye de ella —si el riesgo es asumible, si un modelo local valdría la diferencia, qué se le cuenta al usuario— va en el README: **D11** ya aceptó la dependencia de API, pero la aceptó pesando el catálogo, no las consultas.

---

# L · Versionado del encoder y deriva silenciosa

El modelo forma parte del contrato del índice: los 15.000 vectores solo son comparables entre sí porque salieron del mismo encoder. Mantener esa condición es fácil con pesos locales y **no está garantizado** con un modelo servido por API.

| | Pesos locales | Endpoint gestionado |
|---|---|---|
| Qué fija la versión | el fichero descargado, con su hash | el identificador `gemini-embedding-2`, que el proveedor puede reapuntar |
| Si cambia | no cambia si no se descarga otro | los vectores nuevos dejan de vivir en el mismo espacio que los indexados |
| Cómo se entera uno | — | **por ninguna excepción**: los vecinos simplemente empeoran |

Ese modo de fallo es el que importa: silencioso, y con un síntoma —resultados peores— que se confunde con una representación mediocre. El artefacto de embeddings guarda `model_id`, dimensión, dtype y el SHA-256 del **corpus**, pero no hay huella del **modelo**: la API no devuelve ninguna versión que anotar.

Sí se puede comprobar por sus efectos: recodificar unas fichas ya indexadas y mirar si dan el mismo vector. Con vectores unitarios, coseno ≈ 1 significa que el modelo no ha cambiado.

**Cuántas fichas, y por qué no es un muestreo.** Un canario no estima una proporción: si el proveedor cambia el modelo no cambian el 5 % de los vectores, cambian **todos**, y con una sola ficha se detectaría. El tamaño compra variedad —para no pasar por alto un cambio parcial que mueva unas zonas del espacio y no otras— y sale casi gratis: el **5 % del corpus indexado** son 750 fichas del catálogo completo, unos veinte segundos y menos de una décima de dólar. Un guardián que tarda un minuto acaba saltándose justo antes de los eventos, que es cuando hace falta.

Se toman **las primeras N**, y eso ya da la variedad: el catálogo no está ordenado por ningún atributo, así que las primeras 750 reproducen el conjunto —color vacío al 40,4 % frente al 37,4 % global, marca vacía al 3,7 % frente al 4,4 %, mediana de 1.012 caracteres frente a 936, 675 marcas distintas—. Y son **deterministas**, que es lo que permite compararlas contra los vectores guardados.

In [ ]:
# 📄 DATOS · 🔬 muestra: 75 fichas (el 5 % de 1.500). Sobre el índice real serían
#            750, el 5 % de 📚 catalogo_productos.csv
# El 5 % del corpus indexado: 75 fichas aquí sobre la muestra, 750 sobre el
# catálogo completo, que es donde el canario vigila de verdad (NB08). El suelo
# evita que en un corpus pequeño la comprobación se quede en cuatro fichas.
CANARIO = max(16, round(0.05 * len(corpus_textos)))
TOLERANCIA_DERIVA = 1e-3   # se exige un 99,9 % de parecido con el vector guardado

clave_ganador = ("gemini-2", "sin_contrato")
if clave_ganador in VECTORES:
    encoder = fabricar("gemini-2")
    try:
        deriva = drift_check(
            encoder,
            corpus_textos[:CANARIO],
            VECTORES[clave_ganador]["document"][:CANARIO],
            kind="document", contract="sin_contrato",
            tolerancia=TOLERANCIA_DERIVA,
        )
    finally:
        del encoder
        gc.collect()
    display(pd.DataFrame([deriva]))
    print(
        "✅ El modelo sigue siendo el mismo que indexó el catálogo."
        if deriva["sin_deriva"]
        else "🚨 El encoder ha cambiado: reindexa ANTES de escribir nada más en el índice."
    )
else:
    print("Sin vectores de gemini-2: ejecuta B.2c antes de esta celda.")

### Qué hereda NB04 de esto

La comprobación anterior es un diagnóstico, no una estrategia. La estrategia son dos decisiones del notebook siguiente, al definir el esquema: cuestan cero si se toman entonces y mucho si hay que retrofitarlas.

1. **El nombre de la colección lleva el contrato del índice** — modelo, plantilla y dimensión. Una colección llamada `productos` no dice con qué se codificó; una llamada `productos__gemini2__A4__768` hace imposible ingerir en ella vectores de otro encoder por descuido.
2. **La migración es un cambio de colección, no un borrado.** Reindexar sobre la colección viva deja el buscador respondiendo con un índice a medias; construir la nueva al lado y cambiar el puntero cuando esté completa no tiene ventana de indisponibilidad, y permite volver atrás si algo sale mal.

Y una tercera, de operación: **el canario se ejecuta antes de aplicar los eventos de NB08**, que es el único momento en que se escriben vectores nuevos junto a los viejos. Si el encoder ha cambiado, escribir esas altas contamina el índice en silencio.

In [ ]:
# 📄 DATOS · actualiza artifacts/comparativa_modelos.json
# Los tres criterios de arquitectura que faltaban, al artefacto. Se añaden sobre
# el JSON que dejó la sección G en vez de escribir otro fichero: la comparativa
# de modelos es una sola cosa, y partirla obligaría a cruzarlas para leerla.
destino = Path("..") / "artifacts" / "comparativa_modelos.json"
artefacto = json.loads(destino.read_text(encoding="utf-8"))

artefacto["criterios_arquitectura"] = {
    "consistencia_ciegas": registros(consistencia_modelos),
    "latencia_por_consulta_ms": list(LATENCIAS.values()),
    "coste_api": {
        "precio_por_millon_usd": PRECIO_POR_MILLON,
        "tokenizador_usado": alias_referencia,
        "tokens_catalogo_completo": int(longitudes.sum()),
        "tokens_por_consulta_medio": round(float(tokens_consulta.mean()), 1),
        **coste,
    },
    "deriva_del_encoder": globals().get("deriva"),
}

destino.write_text(
    json.dumps(artefacto, indent=2, ensure_ascii=False, default=str), encoding="utf-8"
)
print(f"Actualizado {destino.name} con criterios_arquitectura "
      f"({destino.stat().st_size / 1024:.1f} KB)")